# 03 - Gemini: model choice, settings, and the test run

**This notebook spends money.** The whole Phase 1 cap is $4.00, enforced in code: a request that could
push total spend over the cap is never sent. Every cell prints an estimate *before* spending and the
total spent afterwards. Rough costs: bake-off up to ~$0.30, batch/thinking checks ~$0.10, test run ~$1-2.
Everything is cached on Drive - rerunning a cell never pays twice for the same complaints.

If a cell says the **daily quota** is used up, rerun that cell tomorrow. If it says the **budget** is reached,
stop: the cap did its job.

**Step 1 - setup (free).** Downloads the project code from GitHub, installs it, connects Google Drive
(all results are saved in `MyDrive/cfpb-nlp-classifier/phase1`), reads your `GEMINI_API_KEY` secret (and the optional `TYPESAFE_API_KEY` Jev secret) and
logs in to Google Sheets. Colab will ask you to allow Drive and Sheets access - click Allow.
Takes 2-4 minutes the first time. If it says the key is missing: click the key icon on the left of
Colab, add a secret named `GEMINI_API_KEY`, and switch "Notebook access" on.

In [ ]:
!git clone -q https://github.com/Dentoneer/cfpb-nlp-classifier /content/cfpb-nlp-classifier || git -C /content/cfpb-nlp-classifier pull -q
%pip install -q -e "/content/cfpb-nlp-classifier[ml,colab]"
import sys; sys.path.insert(0, "/content/cfpb-nlp-classifier")
from nlp.colab import setup
cfg, gc, api_key, jev_key = setup()

**Load data and tools (free).** `tune` is a random 50 of the tuning complaints, used for all the
choices below. The 800 test complaints are not touched until the very end.

In [ ]:
import dataclasses
from nlp import colab, store, pipeline as P
from nlp.llm.classify import run_classification
from nlp.llm.client import GenAIClient
complaints = store.read(cfg, P.STAGE_COMPLAINTS)
taxonomy = store.read(cfg, P.STAGE_TAXONOMY)
splits = store.read(cfg, P.STAGE_SPLITS)
train_df, test_df = P.train_test_frames(complaints, splits)
tuning_df = P.split_rows(complaints, splits, splits["split"] == "tuning")
tune = tuning_df.sample(n=min(50, len(tuning_df)), random_state=cfg.seed)
ledger, gem_cache, judge_cache, limiter = colab.open_tools(cfg)
client = GenAIClient(api_key)

## Part A - Model bake-off

Runs the same 50 tuning complaints through four Gemini models and compares accuracy against the CFPB
labels. **Estimate first (free):** the next cell prints an upper-bound cost per model and stops if the
total will not fit in what is left of the cap.

In [ ]:
CANDIDATES = ["gemini-3.8-flash", "gemini-2.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
est = {m: colab.estimate_classification(cfg, tune, taxonomy, model=m) for m in CANDIDATES}
for m, usd in est.items():
    print(f"{m:24s} up to ${usd:.2f}")
colab.check_fits(ledger, sum(est.values()), "Bake-off")

**Run the bake-off (~$0.30 or less).** **Look at:** the `path` column (all three levels correct) and
`usd` (what each model actually cost). Higher accuracy for the same or lower dollars wins.

In [ ]:
runs = {m: (lambda m=m: run_classification(tune, cfg, client, ledger, gem_cache, f"bakeoff_{m}",
         model=m, taxonomy=taxonomy, limiter=limiter)) for m in CANDIDATES}
bakeoff = colab.compare_runs(runs, tune, ledger)
print(colab.ledger_report(ledger))
bakeoff.round(3)

**Your pick.** Type the winning model in the next cell (replace `None`, keep the quotes) and run it. The
choice is saved on Drive and every later notebook uses it. On a rerun you can leave it as `None`: the
saved pick is kept. The judge model must be a *different* model than the classifier; if you pick
`gemini-3.1-flash-lite`, also set `JUDGE` to another model. Optionally also set `classifier_model` in
`config/phase1.yaml` in the repo so the choice is in git.

In [ ]:
PICK = None    # e.g. "gemini-3.8-flash"
JUDGE = None   # None keeps the current judge model
cfg = colab.apply_choices(cfg, require_pick=True, classifier_model=PICK, judge_model=JUDGE)

## Part B - Batch size and thinking level

Gemini classifies several complaints per request. More per request is cheaper but can be less accurate.
This compares 10 vs 25 per request on the same 50 complaints with your chosen model (~$0.05).
**Look at:** `path` accuracy and `usd`. Keep 10 unless 25 is just as accurate.

In [ ]:
BATCHES = [10, 25]
colab.check_fits(ledger, sum(colab.estimate_classification(cfg, tune, taxonomy, batch_size=b)
                             for b in BATCHES), "Batch-size check")
runs = {f"batch{b}": (lambda b=b: run_classification(tune, dataclasses.replace(cfg, batch_size=b),
        client, ledger, gem_cache, f"batch{b}", taxonomy=taxonomy, limiter=limiter)) for b in BATCHES}
print(colab.compare_runs(runs, tune, ledger).round(3))

**Thinking level.** Gemini can "think" before answering; thinking tokens are billed like output, so
less thinking is cheaper. Pick the *lowest* level that keeps accuracy. **Look at:** `path` and `usd`.

In [ ]:
LEVELS = ["minimal", "low", "medium"]
colab.check_fits(ledger, sum(colab.estimate_classification(cfg, tune, taxonomy, thinking=t)
                             for t in LEVELS), "Thinking check")
runs = {f"think_{t}": (lambda t=t: run_classification(tune, cfg, client, ledger, gem_cache,
        f"think_{t}", thinking=t, taxonomy=taxonomy, limiter=limiter)) for t in LEVELS}
print(colab.compare_runs(runs, tune, ledger).round(3))

**Save your batch size and thinking choices.** Replace `None` with your choice (for example `10` and
`"low"`) and run. `None` keeps the saved value. They apply to every later notebook.

In [ ]:
BATCH_SIZE = None   # 10 or 25
THINKING = None     # "minimal", "low" or "medium"
cfg = colab.apply_choices(cfg, batch_size=BATCH_SIZE, thinking=THINKING)

## Part C (optional) - Prompt iterations

Only if accuracy on the tuning set is disappointing. The prompt lives in code, so a change means asking
Claude to edit `build_prompt` in `nlp/llm/classify.py` and bump `prompt_version` in `config/phase1.yaml`
(`v2`, `v3`...) and push to GitHub. Then **restart the Colab session** (Runtime menu > Restart session),
rerun the setup cell (it pulls the new code; if you skip the restart, setup stops and tells you to
restart, so old-prompt answers are never saved under the new version) and then this cell. Each prompt
version is cached separately. Never tune on the test set.

In [ ]:
name = f"prompt_{cfg.prompt_version}"
colab.check_fits(ledger, colab.estimate_classification(cfg, tune, taxonomy), name)
runs = {name: lambda: run_classification(tune, cfg, client, ledger, gem_cache, name,
        taxonomy=taxonomy, limiter=limiter)}
print(colab.compare_runs(runs, tune, ledger).round(3))

## Part D - The paid test run

**Estimate before spending (free).** Cost of every remaining paid step in notebooks 03 and 04 (test
classification, two self-consistency reruns, the Jev validator, the summary judge). "estimate" uses what the
bake-off and tuning runs really cost per request (x1.5 safety); "bound" is the worst case the spend guard
reserves. The next cell stops if the whole remaining phase will not fit in what is left of the cap. (The
guard in code still refuses any single request that could cross the cap.)

In [ ]:
sc_rows = P.split_rows(complaints, splits, splits["self_consistency"])
est = colab.phase_estimate(cfg, ledger, taxonomy, test_df, sc_rows)
colab.check_fits(ledger, colab.pending_usd(est), "Whole remaining phase",
                 bound_usd=colab.pending_usd(est, column="bound_usd"))

**Classify the 800 test complaints (paid, about $1-2).** Takes 15-40 minutes. If it stops (daily quota
or budget) the message says what to do; finished rows are saved and a rerun continues where it stopped.
**Look at:** accuracy per level vs CFPB labels (compare with the TF-IDF numbers from notebook 02) and the
spend line.

In [ ]:
try:
    gem = colab.stage_paid(cfg, P.STAGE_PRED_GEMINI, lambda: P.run_gemini(
        cfg, test_df, client, ledger, gem_cache, taxonomy, "gemini", limiter),
        test_df["complaint_id"], ledger)
    print({k: round(v, 3) for k, v in colab.accuracy_by_level(gem, test_df).items()})
finally:
    print(colab.ledger_report(ledger))

Done. Next: `04_validators.ipynb`.